In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_DIR)

Project root: d:\ai_project\FinGuide-AI
Processed data: d:\ai_project\FinGuide-AI\data\processed


In [2]:
for filename in [
    "train_sft.json",
    "validation_sft.json",
    "test_sft.json"
]:
    path = PROCESSED_DATA_DIR / filename

    print(
        filename,
        "→",
        "FOUND" if path.exists() else "MISSING"
    )

train_sft.json → FOUND
validation_sft.json → FOUND
test_sft.json → FOUND


In [3]:
import json

with open(PROCESSED_DATA_DIR / "train_sft.json", "r", encoding="utf-8") as f:
    train_sft = json.load(f)

with open(PROCESSED_DATA_DIR / "validation_sft.json", "r", encoding="utf-8") as f:
    validation_sft = json.load(f)

with open(PROCESSED_DATA_DIR / "test_sft.json", "r", encoding="utf-8") as f:
    test_sft = json.load(f)

print(f"Train      : {len(train_sft):,}")
print(f"Validation : {len(validation_sft):,}")
print(f"Test       : {len(test_sft):,}")

Train      : 34,185
Validation : 4,268
Test       : 4,208


In [4]:
print("Fields:", train_sft[0].keys())

print("\nPrompt:")
print(train_sft[0]["prompt"])

print("\nAnswer:")
print(train_sft[0]["answer"])

Fields: dict_keys(['prompt', 'answer'])

Prompt:
### Question
What is the primary difference in impact between parental income and parental wealth regarding a child's educational outcomes?

Answer:
Parental income has a larger effect on whether a child attends college, whereas parental wealth has a significant effect on whether the child graduates from college.


In [6]:
import sys
import importlib.util

print("Python:", sys.version)

packages = [
    "torch",
    "transformers",
    "datasets",
    "peft",
    "trl",
    "bitsandbytes",
    "accelerate",
]

print("\nPackage availability:")

for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package:15} → {'INSTALLED' if installed else 'MISSING'}")

Python: 3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]

Package availability:
torch           → INSTALLED
transformers    → INSTALLED
datasets        → INSTALLED
peft            → INSTALLED
trl             → INSTALLED
bitsandbytes    → INSTALLED
accelerate      → INSTALLED


In [ ]:
import torch
import transformers
import peft
import trl
import accelerate
import pandas

print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("PEFT         :", peft.__version__)
print("TRL          :", trl.__version__)
print("Accelerate   :", accelerate.__version__)

print("\nCUDA available:", torch.cuda.is_available())

d:\ai_project\FinGuide-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch      : 2.14.0+cpu
Transformers : 5.16.1
PEFT         : 0.20.0
TRL          : 1.12.0
Accelerate   : 1.14.0

CUDA available: False


In [8]:
from huggingface_hub import whoami

user_info = whoami()

print("Hugging Face user:", user_info["name"])

Hugging Face user: sandeep1reddy


In [9]:
from transformers import AutoTokenizer

MODEL_ID = "google/gemma-4-E4B-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer loaded successfully")
print("Vocabulary size:", tokenizer.vocab_size)

d:\ai_project\FinGuide-AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mes-Sandeep\.cache\huggingface\hub\models--google--gemma-4-E4B-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer loaded successfully
Vocabulary size: 262144


In [ ]:
text = train_sft[0]["prompt"]

tokens = tokenizer(
    text,
    add_special_tokens=True
)

print("Characters:", len(text))
print("Tokens:", len(tokens["input_ids"]))

print("\nFirst 20 token IDs:")
print(tokens["input_ids"][:20])


Characters: 139
Tokens: 24

First 20 token IDs:
[10354, 19566, 107, 3689, 563, 506, 5905, 4954, 528, 4100, 1534, 49749, 7094, 532, 49749, 12821, 8859, 496, 1919, 236789]


In [12]:
import numpy as np

token_lengths = []

for example in train_sft:
    tokens = tokenizer(
        example["prompt"],
        add_special_tokens=True
    )

    token_lengths.append(len(tokens["input_ids"]))

token_lengths = np.array(token_lengths)

print("Number of examples:", len(token_lengths))
print("Minimum tokens:", token_lengths.min())
print("Maximum tokens:", token_lengths.max())

print("\nPercentiles:")
for p in [50, 75, 90, 95, 99, 99.5]:
    print(f"{p:>5}% :", int(np.percentile(token_lengths, p)))

Number of examples: 34185
Minimum tokens: 8
Maximum tokens: 3327

Percentiles:
   50% : 29
   75% : 41
   90% : 976
   95% : 1203
   99% : 1585
 99.5% : 1818


In [14]:
for limit in [512, 1024, 2048, 4096, 8192]:

    count = np.sum(token_lengths > limit)
    percentage = count / len(token_lengths) * 100

    print(
        f"{limit:>5} tokens → "
        f"{count:>6,} examples "
        f"({percentage:.2f}%) exceed"
    )

  512 tokens →  5,841 examples (17.09%) exceed
 1024 tokens →  2,950 examples (8.63%) exceed
 2048 tokens →    102 examples (0.30%) exceed
 4096 tokens →      0 examples (0.00%) exceed
 8192 tokens →      0 examples (0.00%) exceed


In [15]:
full_lengths = []

for example in train_sft:

    full_text = (
        example["prompt"]
        + "\n\n### Answer\n"
        + example["answer"]
    )

    tokens = tokenizer(
        full_text,
        add_special_tokens=True
    )

    full_lengths.append(len(tokens["input_ids"]))

full_lengths = np.array(full_lengths)

print("Number of examples:", len(full_lengths))
print("Minimum:", full_lengths.min())
print("Maximum:", full_lengths.max())

print("\nPercentiles:")
for p in [50, 75, 90, 95, 99, 99.5]:
    print(f"{p:>5}% :", int(np.percentile(full_lengths, p)))

print("\nExceeding limits:")
for limit in [1024, 2048, 4096]:
    count = np.sum(full_lengths > limit)
    percentage = count / len(full_lengths) * 100

    print(
        f"{limit:>4} → "
        f"{count:,} examples "
        f"({percentage:.2f}%)"
    )

Number of examples: 34185
Minimum: 21
Maximum: 3340

Percentiles:
   50% : 74
   75% : 100
   90% : 984
   95% : 1211
   99% : 1594
 99.5% : 1827

Exceeding limits:
1024 → 3,039 examples (8.89%)
2048 → 103 examples (0.30%)
4096 → 0 examples (0.00%)


In [ ]:
print(tokenizer.chat_template)

In [17]:
example = train_sft[0]

messages = [
    {
        "role": "user",
        "content": example["prompt"]
    },
    {
        "role": "assistant",
        "content": example["answer"]
    }
]

formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text)

<bos><|turn>user
### Question
What is the primary difference in impact between parental income and parental wealth regarding a child's educational outcomes?<turn|>
<|turn>model
Parental income has a larger effect on whether a child attends college, whereas parental wealth has a significant effect on whether the child graduates from college.<turn|>



In [18]:
tokens = tokenizer(
    formatted_text,
    add_special_tokens=False
)

print("Tokens:", len(tokens["input_ids"]))

Tokens: 64


In [19]:
def format_chat_example(example):
    messages = [
        {
            "role": "user",
            "content": example["prompt"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [20]:
formatted_text = format_chat_example(train_sft[0])

print(formatted_text)
print("\n---")
print("Characters:", len(formatted_text))

tokens = tokenizer(
    formatted_text,
    add_special_tokens=False
)

print("Tokens:", len(tokens["input_ids"]))

<bos><|turn>user
### Question
What is the primary difference in impact between parental income and parental wealth regarding a child's educational outcomes?<turn|>
<|turn>model
Parental income has a larger effect on whether a child attends college, whereas parental wealth has a significant effect on whether the child graduates from college.<turn|>


---
Characters: 350
Tokens: 64


In [21]:
chat_lengths = []

for example in train_sft:
    formatted = format_chat_example(example)

    tokens = tokenizer(
        formatted,
        add_special_tokens=False
    )

    chat_lengths.append(len(tokens["input_ids"]))

In [22]:
import numpy as np

print("Min:", np.min(chat_lengths))
print("Max:", np.max(chat_lengths))
print("50%:", np.percentile(chat_lengths, 50))
print("75%:", np.percentile(chat_lengths, 75))
print("90%:", np.percentile(chat_lengths, 90))
print("95%:", np.percentile(chat_lengths, 95))
print("99%:", np.percentile(chat_lengths, 99))
print("99.5%:", np.percentile(chat_lengths, 99.5))

print(">1024:", sum(x > 1024 for x in chat_lengths))
print(">2048:", sum(x > 2048 for x in chat_lengths))

Min: 28
Max: 3347
50%: 81.0
75%: 107.0
90%: 991.6000000000022
95%: 1218.0
99%: 1601.0
99.5%: 1834.0
>1024: 3103
>2048: 104


In [23]:
MAX_SEQ_LENGTH = 2048

def tokenize_sft_example(example):
    messages = [
        {
            "role": "user",
            "content": example["prompt"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    # Full conversation
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # User portion only
    user_text = tokenizer.apply_chat_template(
        [messages[0]],
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize full conversation
    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )

    # Tokenize user portion to determine where assistant starts
    user_tokens = tokenizer(
        user_text,
        add_special_tokens=False
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    user_length = min(len(user_tokens["input_ids"]), len(input_ids))

    # Ignore user tokens when calculating loss
    labels = [-100] * user_length + input_ids[user_length:]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [24]:
example = train_sft[0]

tokenized = tokenize_sft_example(example)

print("Input IDs:", len(tokenized["input_ids"]))
print("Attention mask:", len(tokenized["attention_mask"]))
print("Labels:", len(tokenized["labels"]))

print("\nFirst 30 labels:")
print(tokenized["labels"][:30])

print("\nLast 30 labels:")
print(tokenized["labels"][-30:])

Input IDs: 64
Attention mask: 64
Labels: 64

First 30 labels:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

Last 30 labels:
[514, 7094, 815, 496, 6268, 1763, 580, 3363, 496, 1919, 93525, 8672, 236764, 12176, 49749, 12821, 815, 496, 3629, 1763, 580, 3363, 506, 1919, 36215, 699, 8672, 236761, 106, 107]


In [25]:
for i, (input_id, label) in enumerate(
    zip(tokenized["input_ids"], tokenized["labels"])
):
    token = tokenizer.decode([input_id])

    if label != -100:
        print("First trainable token position:", i)
        print("Token:", repr(token))
        print("Token ID:", input_id)
        break

First trainable token position: 33
Token: 'Parent'
Token ID: 18674


In [26]:
print("Masked tokens:", tokenized["labels"].count(-100))
print("Trainable tokens:", sum(x != -100 for x in tokenized["labels"]))

Masked tokens: 33
Trainable tokens: 31


In [27]:
from datasets import Dataset

def tokenize_dataset(sft_examples):
    dataset = Dataset.from_list(sft_examples)

    return dataset.map(
        tokenize_sft_example,
        remove_columns=dataset.column_names,
        desc="Tokenizing dataset"
    )

In [28]:
train_tokenized = tokenize_dataset(train_sft)
validation_tokenized = tokenize_dataset(validation_sft)
test_tokenized = tokenize_dataset(test_sft)

Tokenizing dataset: 100%|██████████| 4208/4208 [00:06<00:00, 617.00 examples/s]


In [29]:
print(train_tokenized)
print(train_tokenized.column_names)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 34185
})
['input_ids', 'attention_mask', 'labels']


In [30]:
for i in range(3):
    example = train_tokenized[i]

    print(f"\nExample {i}")
    print("Input tokens:", len(example["input_ids"]))
    print("Masked tokens:", example["labels"].count(-100))
    print(
        "Trainable tokens:",
        sum(label != -100 for label in example["labels"])
    )


Example 0
Input tokens: 64
Masked tokens: 33
Trainable tokens: 31

Example 1
Input tokens: 139
Masked tokens: 69
Trainable tokens: 70

Example 2
Input tokens: 94
Masked tokens: 34
Trainable tokens: 60


In [31]:
max_length = max(
    len(example["input_ids"])
    for example in train_tokenized
)

print("Maximum tokenized length:", max_length)

Maximum tokenized length: 2048


In [33]:
TOKENIZED_DATA_DIR = PROJECT_ROOT / "data" / "tokenized"
TOKENIZED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [34]:
train_tokenized.save_to_disk(
    str(TOKENIZED_DATA_DIR / "train")
)

validation_tokenized.save_to_disk(
    str(TOKENIZED_DATA_DIR / "validation")
)

test_tokenized.save_to_disk(
    str(TOKENIZED_DATA_DIR / "test")
)

print("Tokenized datasets saved successfully.")

Saving the dataset (1/1 shards): 100%|██████████| 4208/4208 [00:00<00:00, 118668.94 examples/s]

Tokenized datasets saved successfully.


In [35]:
from datasets import load_from_disk

train_check = load_from_disk(
    str(TOKENIZED_DATA_DIR / "train")
)

validation_check = load_from_disk(
    str(TOKENIZED_DATA_DIR / "validation")
)

test_check = load_from_disk(
    str(TOKENIZED_DATA_DIR / "test")
)

print("Train:", len(train_check))
print("Validation:", len(validation_check))
print("Test:", len(test_check))

print("\nColumns:", train_check.column_names)

Train: 34185
Validation: 4268
Test: 4208

Columns: ['input_ids', 'attention_mask', 'labels']


#### QLoRA fine-tuning strategy

In [44]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(MODEL_ID)

print(config.model_type)
print(config.architectures)

gemma4
['Gemma4ForConditionalGeneration']


In [48]:
list(config)

['transformers_version',
 'architectures',
 'output_hidden_states',
 'return_dict',
 'dtype',
 'chunk_size_feed_forward',
 'is_encoder_decoder',
 'id2label',
 'label2id',
 'problem_type',
 'text_config',
 'vision_config',
 'audio_config',
 'boi_token_id',
 'eoi_token_id',
 'image_token_id',
 'video_token_id',
 'boa_token_id',
 'eoa_token_index',
 'audio_token_id',
 'initializer_range',
 'tie_word_embeddings',
 '_name_or_path',
 '_commit_hash',
 '_output_attentions',
 '_attn_implementation_internal',
 '_experts_implementation_internal',
 'eoa_token_id',
 'eos_token_id',
 'model_type',
 'vision_soft_tokens_per_image']

In [50]:
from transformers import AutoModelForCausalLM

print(AutoModelForCausalLM)

<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>


In [51]:
from transformers import Gemma4ForConditionalGeneration
import inspect

print(inspect.getsource(Gemma4ForConditionalGeneration)[:5000])

@auto_docstring(
    custom_intro="""
    The base Gemma 4 model comprising a vision backbone, an audio backbone, a language model, and a language modeling
    head.
    """
)
class Gemma4ForConditionalGeneration(Gemma4PreTrainedModel, GenerationMixin):
    _tied_weights_keys = {"lm_head.weight": "model.language_model.embed_tokens.weight"}
    accepts_loss_kwargs = False
    base_model_prefix = "model"

    def __init__(self, config: Gemma4Config):
        super().__init__(config)
        self.model = Gemma4Model(config)
        self.lm_head = nn.Linear(config.text_config.hidden_size, config.text_config.vocab_size, bias=False)
        self.post_init()

    @auto_docstring
    def get_image_features(
        self,
        pixel_values: torch.FloatTensor,
        image_position_ids: torch.LongTensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ):
        r"""
        image_position_ids (`torch.LongTensor` of shape `(batch_size, max_patches, 2)`, *optional*):
         

In [52]:
from transformers.models.gemma4.modeling_gemma4 import Gemma4TextModel

import inspect

source = inspect.getsource(Gemma4TextModel)

print(source[:12000])

@auto_docstring(custom_intro="The base Gemma 4 language model without a language modeling head.")
class Gemma4TextModel(Gemma4PreTrainedModel):
    config: Gemma4TextConfig
    input_modalities = ("text",)
    _can_record_outputs = {
        "router_logits": OutputRecorder(Gemma4TextRouter, index=0),
        "hidden_states": Gemma4TextDecoderLayer,
        "attentions": Gemma4TextAttention,
    }

    def __init__(self, config: Gemma4TextConfig):
        super().__init__(config)
        self.padding_idx = config.pad_token_id
        self.vocab_size = config.vocab_size

        # Gemma4 downcasts the below to bfloat16, causing sqrt(3072)=55.4256 to become 55.5. See https://github.com/huggingface/transformers/pull/29402
        self.embed_tokens = Gemma4TextScaledWordEmbedding(
            config.vocab_size, config.hidden_size, self.padding_idx, embed_scale=self.config.hidden_size**0.5
        )
        self.layers = nn.ModuleList(
            [Gemma4TextDecoderLayer(config, layer_idx) f

In [53]:
from transformers.models.gemma4.modeling_gemma4 import Gemma4TextAttention
import inspect

source = inspect.getsource(Gemma4TextAttention)

print(source)

class Gemma4TextAttention(nn.Module):
    """Multi-headed attention from 'Attention Is All You Need' paper"""

    def __init__(self, config: Gemma4TextConfig, layer_idx: int):
        super().__init__()
        self.layer_type = config.layer_types[layer_idx] if hasattr(config, "layer_types") else None
        self.config = config
        self.layer_idx = layer_idx
        self.is_sliding = self.layer_type == "sliding_attention"
        self.sliding_window = config.sliding_window if self.is_sliding else None

        layer_config = config.per_layer_config[layer_idx]
        self.head_dim = layer_config.head_dim
        self.use_alternative_attention = config.attention_k_eq_v and not self.is_sliding
        self.num_key_value_groups = config.num_attention_heads // layer_config.num_key_value_heads
        self.scaling = 1.0
        self.attention_dropout = self.config.attention_dropout
        self.is_causal = config.use_bidirectional_attention != "all"

        # Shared kv cache
       

In [54]:
from transformers.models.gemma4.modeling_gemma4 import Gemma4TextMLP
import inspect

print(inspect.getsource(Gemma4TextMLP))

class Gemma4TextMLP(nn.Module):
    def __init__(self, config: Gemma4TextConfig, layer_idx: int):
        super().__init__()
        first_kv_shared_layer_idx = config.num_hidden_layers - config.num_kv_shared_layers
        is_kv_shared_layer = layer_idx >= first_kv_shared_layer_idx > 0
        use_double_wide_mlp = config.use_double_wide_mlp and is_kv_shared_layer
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size * (2 if use_double_wide_mlp else 1)
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=False)
        self.act_fn = ACT2FN[config.hidden_activation]

    def forward(self, x):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        return down_proj



In [55]:
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
}

In [56]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

<pre>

QLoRA
│
├── Base model
│     ├── 4-bit
│     ├── NF4
│     ├── double quantization
│     └── frozen
│
└── LoRA
      ├── r = 16
      ├── alpha = 32
      ├── dropout = 0.05
      └── attention + FFN projections

</pre>

batch size → gradient accumulation → learning rate → epochs → checkpointing → evaluation frequency

In [57]:
TRAINING_CONFIG = {
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "num_train_epochs": 2,
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "cosine",
    "gradient_checkpointing": True,
    "weight_decay": 0.01,
}

<pre>
MODEL
Gemma 4 E4B IT
        │
        ▼
QUANTIZATION
4-bit NF4
BF16 compute
Double quantization
        │
        ▼
LoRA
r = 16
alpha = 32
dropout = 0.05
        │
        ├── q_proj
        ├── k_proj
        ├── v_proj
        ├── o_proj
        ├── gate_proj
        ├── up_proj
        └── down_proj
        │
        ▼
TRAINING
Batch = 1
Accumulation = 8
LR = 2e-4
Epochs = 2
Warmup = 5%
Cosine decay
Gradient checkpointing
Weight decay = 0.01

<pre>

                    PHASE 3
                       │
        ┌──────────────┴──────────────┐
        ▼                             ▼
   QLoRA design                 Training design
        │                             │
   4-bit NF4                     batch = 1*
   BF16                          accum = 8*
   double quant                  LR = 2e-4
        │                         epochs = 2
   LoRA r = 16                   warmup = 5%
   alpha = 32                    cosine
   dropout = .05                 checkpointing
        │                         weight decay
   7 target modules
        │
        └──────────────┬──────────────┘
                       ▼
                GPU requirement
                       │
                       ▼
                 AWS cost estimate

<pre>
QLoRA design       ✅
LoRA targets       ✅
LoRA r/alpha       ✅
Quantization       ✅
Training config    ✅
GPU candidate      g5.xlarge
                  ↓
             NEXT: GPU smoke test

<pre>
LAPTOP
  │
  ├── Dataset preparation ✅
  ├── Tokenization ✅
  ├── QLoRA configuration
  └── Training code preparation
          │
          ▼
       AWS EC2 GPU
          │
          ├── Smoke test
          │     ├── Load Gemma 4
          │     ├── 4-bit NF4
          │     ├── Attach LoRA
          │     ├── Forward
          │     ├── Backward
          │     └── Measure VRAM
          │
          └── If smoke test passes
                ↓
             Actual QLoRA
             fine-tuning